# Method 1 - Historical-Reproduction Based Feature Construction and Neural Network Prediction

## 1. Import Libraries

In [1]:
import os
import math
import numpy as np
import pandas as pd
import networkx as nx
from itertools import combinations
from joblib import Parallel, delayed
from tqdm import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

## 2. Constract the Historical-Reproduction Based Features

### 2.1 Load the Processed Data

In [2]:
PATH_PREFIX = '../'

data_tran = pd.read_json(PATH_PREFIX + 'data/data_processed/data_tran.json', orient='index')
data_test = pd.read_json(PATH_PREFIX + 'data/data_processed/data_test.json', orient='index')

# data_tran = data_tran.head(200)
# data_test = data_test.head(50)

In [3]:
n_tran = data_tran.shape[0]
n_test = data_test.shape[0]

In [4]:
data_tran.head(3)

,authors,authors_main,authors_coop,venue,year,list_title,list_abstract,list_combined,str_title,str_abstract,str_combined
0,"[42, 13720, 36]","[42, 36]",[13720],20,9,"[41, 1550, 1563, 1594, 1544, 1919, 1644, 37, 1...","[2455, 1858, 2335, 1543, 1800, 1860, 2000, 286...","[41, 1550, 1563, 1594, 1544, 1919, 1644, 37, 1...",41 1550 1563 1594 1544 1919 1644 37 1539 1715 ...,2455 1858 2335 1543 1800 1860 2000 2867 1546 1...,41 1550 1563 1594 1544 1919 1644 37 1539 1715 ...
1,"[1359, 15881, 45]",[45],"[1359, 15881]",2,15,"[1731, 47, 11, 57, 4624, 1525, 1535, 47, 11, 3...","[40, 1542, 1691, 2449, 1535, 3616, 2206, 1904,...","[1731, 47, 11, 57, 4624, 1525, 1535, 47, 11, 3...",1731 47 11 57 4624 1525 1535 47 11 3522 2223 1653,40 1542 1691 2449 1535 3616 2206 1904 1642 154...,1731 47 11 57 4624 1525 1535 47 11 3522 2223 1...
2,"[19166, 17763]",[-1],"[19166, 17763]",465,17,"[2085, 1719, 1846, 1745, 2243, 1553, 1606, 159...","[40, 1542, 1691, 2449, 1535, 2610, 1543, 1535,...","[2085, 1719, 1846, 1745, 2243, 1553, 1606, 159...",2085 1719 1846 1745 2243 1553 1606 1596 42 41 ...,40 1542 1691 2449 1535 2610 1543 1535 1984 196...,2085 1719 1846 1745 2243 1553 1606 1596 42 41 ...


In [5]:
data_test.head(3)

,authors_coop,venue,year,list_title,list_abstract,list_combined,str_title,str_abstract,str_combined
0,"[16336, 1762, 4357, 12564]",223,19,"[3207, 24, 1798, 1738, 37, 2375, 1568, 11, 53,...","[37, 1662, 3207, 10, 33, 2037, 1738, 1642, 155...","[3207, 24, 1798, 1738, 37, 2375, 1568, 11, 53,...",3207 24 1798 1738 37 2375 1568 11 53 1584 1903...,37 1662 3207 10 33 2037 1738 1642 1553 4917 11...,3207 24 1798 1738 37 2375 1568 11 53 1584 1903...
1,"[21189, 14088]",223,19,"[40, 1560, 1536, 1544, 1609, 1705, 1658, 1543,...","[1731, 2130, 3674, 1705, 1656, 3077, 1546, 367...","[40, 1560, 1536, 1544, 1609, 1705, 1658, 1543,...",40 1560 1536 1544 1609 1705 1658 1543 52 11 33...,1731 2130 3674 1705 1656 3077 1546 3675 2051 2...,40 1560 1536 1544 1609 1705 1658 1543 52 11 33...
2,"[3625, 1198, 19889, 794, 2749, 7801]",7,19,"[47, 1574, 1729, 1641, 11, 37, 2533, 2015, 47,...","[1551, 1728, 3920, 1542, 1535, 1656, 1543, 153...","[47, 1574, 1729, 1641, 11, 37, 2533, 2015, 47,...",47 1574 1729 1641 11 37 2533 2015 47 1930 1549...,1551 1728 3920 1542 1535 1656 1543 1530 3053 2...,47 1574 1729 1641 11 37 2533 2015 47 1930 1549...


### 2.2 Feature Engineering on Co-authorship Network 

In [6]:
def get_coauthors_graph(data):

    coauthors_graph = nx.Graph()

    for authors in data['authors']:
        for author_pair in combinations(authors, 2):
            if coauthors_graph.has_edge(*author_pair):
                coauthors_graph[author_pair[0]][author_pair[1]]['weight'] += 1
            else:
                coauthors_graph.add_edge(author_pair[0], author_pair[1], weight=1)
    
    return coauthors_graph

coauthors_graph = get_coauthors_graph(data_tran)

In [7]:
def get_coauthors_vector(graph, start_nodes):

    node_array = np.zeros((1, 100)) 

    def dfs_iterative(graph, start_node):
        
        if start_node not in graph:
            return
        
        stack = [(start_node, 1)] 
        visited = set() 

        while stack:
            
            node, depth = stack.pop()

            if node in visited:
                continue 

            visited.add(node)

            for neighbor, edge in graph[node].items():
                weight = edge['weight'] * (1 / (depth * math.log(depth + 1)))
                if 0 <= neighbor < 100: 
                    node_array[0, neighbor] += weight 
                if neighbor not in visited:
                    stack.append((neighbor, depth + 1)) 

    for start_node in start_nodes:
        dfs_iterative(graph, start_node)

    return node_array.reshape(1, 100)

In [8]:
def get_coauthors_matrix(data, graph):
    vectors_list = Parallel(n_jobs=-1)(delayed(get_coauthors_vector)(graph, row['authors_coop']) for _, row in tqdm(data.iterrows(), total=len(data)))
    return np.concatenate(vectors_list, axis=0)

In [9]:
if os.path.exists(PATH_PREFIX + 'data/data_processed/x_tran_coauthors_m1.npy'):
    x_tran_coauthors = np.load(PATH_PREFIX + 'data/data_processed/x_tran_coauthors_m1.npy')
else:
    x_tran_coauthors = get_coauthors_matrix(data_tran, coauthors_graph)
    np.save(PATH_PREFIX + 'data/data_processed/x_tran_coauthors_m1.npy', x_tran_coauthors)

if os.path.exists(PATH_PREFIX + 'data/data_processed/x_test_coauthors_m1.npy'):
    x_test_coauthors = np.load(PATH_PREFIX + 'data/data_processed/x_test_coauthors_m1.npy')
else:
    x_test_coauthors = get_coauthors_matrix(data_test, coauthors_graph)
    np.save(PATH_PREFIX + 'data/data_processed/x_test_coauthors_m1.npy', x_test_coauthors)

### 2.3 Feature Engineering on Publication Venues

#### 2.3.1 Feature Engineering on Publication Venues based on Venue Histories

In [10]:
def get_venue_dict_a(data):
    
    num_venues=466
    vector_size=21246

    venue_dict = {venue: np.zeros(vector_size, dtype=int) for venue in range(num_venues)}

    for _, row in tqdm(data.iterrows(), total=len(data)):
        venue = row['venue'] 
        authors = row['authors'] 

        for author_id in authors:
            if author_id < 21246: 
                venue_dict[venue][author_id] += 1

    return venue_dict

venue_dict_a = get_venue_dict_a(data_tran)

100%|██████████| 25793/25793 [00:00<00:00, 27159.87it/s]


In [11]:
def get_venue_array_a(venue, venue_dict):
    return np.array(venue_dict[venue][:100]).reshape(1, 100)

In [12]:
def get_venue_matrix_a(data, venue_dict):
    vectors_list = Parallel(n_jobs=-1)(delayed(get_venue_array_a)(row['venue'], venue_dict) for _, row in tqdm(data.iterrows(), total=len(data)))
    return np.concatenate(vectors_list, axis=0)

In [13]:
if os.path.exists(PATH_PREFIX + 'data/data_processed/x_tran_venue_a_m1.npy'):
    x_tran_venue_a = np.load(PATH_PREFIX + 'data/data_processed/x_tran_venue_a_m1.npy')
else:
    x_tran_venue_a = get_venue_matrix_a(data_tran, venue_dict_a)
    np.save(PATH_PREFIX + 'data/data_processed/x_tran_venue_a_m1.npy', x_tran_venue_a)

if os.path.exists(PATH_PREFIX + 'data/data_processed/x_test_venue_a_m1.npy'):
    x_test_venue_a = np.load(PATH_PREFIX + 'data/data_processed/x_test_venue_a_m1.npy')
else:
    x_test_venue_a = get_venue_matrix_a(data_test, venue_dict_a)
    np.save(PATH_PREFIX + 'data/data_processed/x_test_venue_a_m1.npy', x_test_venue_a)

#### 2.3.2 Feature Engineering on Publication Venues based on Venue Histories & Coauthorship Network

In [14]:
def get_venue_dict_b(data):
    
    num_venues=466
    vector_size=21246

    venue_dict = {venue: np.zeros(vector_size, dtype=int) for venue in range(num_venues)}

    for _, row in tqdm(data.iterrows(), total=len(data)):
        venue = row['venue'] 
        authors = row['authors'] 

        for author_id in authors:
            if author_id < 21246: 
                venue_dict[venue][author_id] += 1

    return venue_dict

venue_dict_b = get_venue_dict_b(data_tran)

100%|██████████| 25793/25793 [00:00<00:00, 29576.61it/s]


In [15]:
def get_venue_vector_b(coauthor_list, venue_dict):

    temp_array = np.zeros(100) 

    for author in coauthor_list:
        relevant_venues = [venue for venue, array in venue_dict.items() if array[author] > 0]
        for venue in relevant_venues:
            temp_array += venue_dict[venue][:100]

    return temp_array.reshape(1, 100)

In [16]:
def get_venue_matrix_b(data, venue_dict):
    vectors_list = Parallel(n_jobs=-1)(delayed(get_venue_vector_b)(row['authors_coop'], venue_dict) for _, row in tqdm(data.iterrows(), total=len(data)))
    return np.concatenate(vectors_list, axis=0)

In [17]:
if os.path.exists(PATH_PREFIX + 'data/data_processed/x_tran_venue_b_m1.npy'):
    x_tran_venue_b = np.load(PATH_PREFIX + 'data/data_processed/x_tran_venue_b_m1.npy')
else:
    x_tran_venue_b = get_venue_matrix_b(data_tran, venue_dict_b)
    np.save(PATH_PREFIX + 'data/data_processed/x_tran_venue_b_m1.npy', x_tran_venue_b)

if os.path.exists(PATH_PREFIX + 'data/data_processed/x_test_venue_b_m1.npy'):
    x_test_venue_b = np.load(PATH_PREFIX + 'data/data_processed/x_test_venue_b_m1.npy')
else:
    x_test_venue_b = get_venue_matrix_b(data_test, venue_dict_b)
    np.save(PATH_PREFIX + 'data/data_processed/x_test_venue_b_m1.npy', x_test_venue_b)

### 2.4 Feature Engineering on Text

#### 2.4.1 Feature Engineering on Text Histories

In [18]:
tfidf_encoder_a = TfidfVectorizer(ngram_range=(1, 2))
tfidf_encoder_a.fit(data_tran['str_combined']) 

def get_top_tfidf_words_a(text, top_n):
    tfidf_array = tfidf_encoder_a.transform([text]).toarray()[0]
    top_indices = np.argsort(tfidf_array)[-top_n:][::-1] 
    return top_indices

In [19]:
def get_word_author_dict_a(data, top_n):

    num_authors = 21246
    word_author_dict = {author_id: {} for author_id in range(num_authors)}

    for _, row in tqdm(data.iterrows(), total=len(data)):
        text = row['str_combined'] 
        authors = row['authors'] 

        top_words = get_top_tfidf_words_a(text, top_n) 

        for author in authors:
            if author >= 0: 
                for word_id in top_words:
                    if word_id in word_author_dict[author]:
                        word_author_dict[author][word_id] += 1
                    else:
                        word_author_dict[author][word_id] = 1

    return word_author_dict

word_author_dict_a = get_word_author_dict_a(data_tran, top_n=10)

100%|██████████| 25793/25793 [11:19<00:00, 37.93it/s]


In [20]:
def get_text_vector_a(text, word_author_dict):

    result_array = np.zeros(100) 
    top_words = get_top_tfidf_words_a(text, top_n=20) 

    for main_author in range(100):
        if main_author not in word_author_dict:
            continue 

        for word in top_words:
            if word in word_author_dict[main_author]:
                result_array[main_author] += word_author_dict[main_author][word]

    return result_array.reshape(1, 100)

In [21]:
def get_text_matrix_a(data, word_author_dict):
    vectors_list = Parallel(n_jobs=-1)(delayed(get_text_vector_a)(row['str_combined'], word_author_dict) for _, row in tqdm(data.iterrows(), total=len(data)))
    return np.concatenate(vectors_list, axis=0)

In [22]:
if os.path.exists(PATH_PREFIX + 'data/data_processed/x_tran_text_a_m1.npy'):
    x_tran_text_a = np.load(PATH_PREFIX + 'data/data_processed/x_tran_text_a_m1.npy')
else:
    x_tran_text_a = get_text_matrix_a(data_tran, word_author_dict_a)
    np.save(PATH_PREFIX + 'data/data_processed/x_tran_text_a_m1.npy', x_tran_text_a)

if os.path.exists(PATH_PREFIX + 'data/data_processed/x_test_text_a_m1.npy'):
    x_test_text_a = np.load(PATH_PREFIX + 'data/data_processed/x_test_text_a_m1.npy')
else:
    x_test_text_a = get_text_matrix_a(data_test, word_author_dict_a)
    np.save(PATH_PREFIX + 'data/data_processed/x_test_text_a_m1.npy', x_test_text_a)

#### 2.4.2 Feature Engineering on Text Histories & Coauthorship Network

In [23]:
tfidf_encoder_b = TfidfVectorizer(ngram_range=(1, 2))
tfidf_encoder_b.fit(data_tran['str_combined']) 

def get_top_tfidf_words_b(text, top_n):
    tfidf_array = tfidf_encoder_b.transform([text]).toarray()[0]
    top_indices = np.argsort(tfidf_array)[-top_n:][::-1] 
    return top_indices

In [24]:
def get_word_author_dict_b(data, top_n):

    num_authors = 21246
    word_author_dict = {author_id: {} for author_id in range(num_authors)}

    for _, row in tqdm(data.iterrows(), total=len(data)):
        text = row['str_combined'] 
        authors = row['authors'] 

        top_words = get_top_tfidf_words_b(text, top_n) 

        for author in authors:
            if author >= 0: 
                for word_id in top_words:
                    if word_id in word_author_dict[author]:
                        word_author_dict[author][word_id] += 1
                    else:
                        word_author_dict[author][word_id] = 1

    return word_author_dict

word_author_dict_b = get_word_author_dict_b(data_tran, top_n=10)

100%|██████████| 25793/25793 [11:36<00:00, 37.05it/s]


In [25]:
def get_text_vector_b(coauthor_list, word_author_dict):

    result_array = np.zeros(100)

    for coauthor in coauthor_list:
        if coauthor not in word_author_dict:
            continue 

        common_words = word_author_dict[coauthor].keys()
        
        for main_author in range(100):
            if main_author not in word_author_dict:
                continue 
            
            for word in common_words:
                if word in word_author_dict[main_author]:
                    result_array[main_author] += word_author_dict[main_author][word]

    return result_array.reshape(1, 100)

In [26]:
def get_text_matrix_b(data, word_author_dict):
    vectors_list = Parallel(n_jobs=-1)(delayed(get_text_vector_b)(row['authors_coop'], word_author_dict) for _, row in tqdm(data.iterrows(), total=len(data)))
    return np.concatenate(vectors_list, axis=0)

In [27]:
if os.path.exists(PATH_PREFIX + 'data/data_processed/x_tran_text_b_m1.npy'):
    x_tran_text_b = np.load(PATH_PREFIX + 'data/data_processed/x_tran_text_b_m1.npy')
else:
    x_tran_text_b = get_text_matrix_b(data_tran, word_author_dict_b)
    np.save(PATH_PREFIX + 'data/data_processed/x_tran_text_b_m1.npy', x_tran_text_b)

if os.path.exists(PATH_PREFIX + 'data/data_processed/x_test_text_b_m1.npy'):
    x_test_text_b = np.load(PATH_PREFIX + 'data/data_processed/x_test_text_b_m1.npy')
else:
    x_test_text_b = get_text_matrix_b(data_test, word_author_dict_b)
    np.save(PATH_PREFIX + 'data/data_processed/x_test_text_b_m1.npy', x_test_text_b)

### 2.5 Combine All Features

In [28]:
y_tran = np.load(PATH_PREFIX + "/data/data_processed/y_tran.npy")
y_tran = y_tran[:n_tran]

x_tran = np.concatenate((x_tran_coauthors, x_tran_venue_a, x_tran_venue_b, x_tran_text_a, x_tran_text_b), axis=1)
x_tran, x_vald, y_tran, y_vald = train_test_split(x_tran, y_tran, test_size=0.2, random_state=42)
x_test = np.concatenate((x_test_coauthors, x_test_venue_a, x_test_venue_b, x_test_text_a, x_test_text_b), axis=1)

## 3. Neural Network Prediction Model

### 3.1 Create the Dataset & DataLoader

In [29]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [30]:
y_tran = torch.tensor(y_tran, dtype=torch.float32).to(device)
x_tran = torch.tensor(x_tran, dtype=torch.float32).to(device)

y_vald = torch.tensor(y_vald, dtype=torch.float32).to(device)
x_vald = torch.tensor(x_vald, dtype=torch.float32).to(device)

x_test = torch.tensor(x_test, dtype=torch.float32).to(device)

### 3.2 Construct the Neural Network Model

In [31]:
class FNN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(FNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, output_dim)
        self.tanh = nn.Tanh() 
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid() 
        self.dropout = nn.Dropout(0.1) 

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc3(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc4(x)
        x = self.sigmoid(x) 
        return x

### 3.3 Train the Neural Network Model

In [32]:
model = FNN(input_dim=x_tran.shape[1], output_dim=y_tran.shape[1]).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.00001, weight_decay=0.00001)

In [33]:
def calculate_metrics(pred_label, true_label):
    pred_label = pred_label.int()
    true_label = true_label.int()
    pc = precision_score(true_label.cpu(), pred_label.cpu(), average='macro', zero_division=0)
    rc = recall_score(true_label.cpu(), pred_label.cpu(), average='macro', zero_division=0)
    f1 = f1_score(true_label.cpu(), pred_label.cpu(), average='macro', zero_division=0)
    return pc, rc, f1

In [34]:
class EarlyStopping:
    def __init__(self, patience=10, delta=0.001):
        self.patience = patience
        self.delta = delta
        self.best_loss = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, train_loss):
        if self.best_loss is None or train_loss < self.best_loss - self.delta:
            self.best_loss = train_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

early_stopping = EarlyStopping(patience=10, delta=0.001)

In [35]:
epochs = 50000

for epoch in range(epochs):
    
    model.train()

    optimizer.zero_grad()

    y_tran_pred_prob = model(x_tran)
    loss = criterion(y_tran_pred_prob, y_tran.float())

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 500 == 0:
        model.eval()

        with torch.no_grad():
            y_tran_pred_prob = model(x_tran)
            y_tran_pred_labl = (y_tran_pred_prob > 0.5).int()
            tran_pc, tran_rc, tran_f1 = calculate_metrics(y_tran_pred_labl, y_tran)

        with torch.no_grad():
            y_vald_pred_prob = model(x_vald)
            y_vald_pred_labl = (y_vald_pred_prob > 0.5).int()
            vald_pc, vald_rc, vald_f1 = calculate_metrics(y_vald_pred_labl, y_vald)

        print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.4f}")
        print(f"Train - Precision: {tran_pc:.4f}, Recall: {tran_rc:.4f}, F1 Score: {tran_f1:.4f}")
        print(f"Val   - Precision: {vald_pc:.4f}, Recall: {vald_rc:.4f}, F1 Score: {vald_f1:.4f}")
        print()

        early_stopping(loss)
        if early_stopping.early_stop:
            print("Early Stop !")
            break

Epoch 500/50000, Loss: 0.1138
Train - Precision: 0.0094, Recall: 0.0105, F1 Score: 0.0091
Val   - Precision: 0.0188, Recall: 0.0108, F1 Score: 0.0098

Epoch 1000/50000, Loss: 0.0627
Train - Precision: 0.0075, Recall: 0.0098, F1 Score: 0.0085
Val   - Precision: 0.0073, Recall: 0.0098, F1 Score: 0.0084

Epoch 1500/50000, Loss: 0.0498
Train - Precision: 0.0078, Recall: 0.0096, F1 Score: 0.0086
Val   - Precision: 0.0076, Recall: 0.0096, F1 Score: 0.0085

Epoch 2000/50000, Loss: 0.0429
Train - Precision: 0.0460, Recall: 0.0139, F1 Score: 0.0161
Val   - Precision: 0.0277, Recall: 0.0138, F1 Score: 0.0155

Epoch 2500/50000, Loss: 0.0375
Train - Precision: 0.1361, Recall: 0.0298, F1 Score: 0.0417
Val   - Precision: 0.1186, Recall: 0.0293, F1 Score: 0.0389

Epoch 3000/50000, Loss: 0.0323
Train - Precision: 0.2650, Recall: 0.0663, F1 Score: 0.0945
Val   - Precision: 0.2101, Recall: 0.0568, F1 Score: 0.0800

Epoch 3500/50000, Loss: 0.0282
Train - Precision: 0.3672, Recall: 0.1206, F1 Score: 0.167

### 3.4 Predict on the Test Set

In [36]:
with torch.no_grad():
    y_test_pred_prob = model(x_test)
    y_test_pred_labl = (y_test_pred_prob > 0.5).int()

In [37]:
def generate_output_csv(x_test, y_test_pred_labl):
    
    result = []
    
    for i, row in enumerate(y_test_pred_labl):
        if ((x_test[i, :100] < 1).all() or (x_test[i, 100:200] == 0).all() or (x_test[i, 200:300] == 0).all() or (x_test[i, 300:400] == 0).all() or (x_test[i, 400:500] == 0).all()):
            result.append("-1")
        elif row.sum() == 0 or row[100] == 1:
            result.append("-1")
        else:
            indices = [str(idx) for idx, val in enumerate(row) if val == 1]
            result.append(" ".join(indices))
    
    result_df = pd.DataFrame({"ID": range(len(result)), "Predict": result})
    
    return result_df

generate_output_csv(x_test, y_test_pred_labl).to_csv("../data/data_result/result_method1.csv", index=False)